In [5]:
## create a binary tensor with indicators for extreme events, uncomment to save as npz files.

import numpy as np
import os

# Define the percentile
xx = 95

base = r"D:\omer\Hyb-STEX\data"
save_path_base = r"D:\omer\Hyb-STEX\data_with_EVs"
dataset = "BJTaxi"

os.makedirs(os.path.join(save_path_base, dataset), exist_ok=True)

for split in ["train", "val", "test"]:
    data_path = os.path.join(base, dataset, f"{split}.npz")
    out_path = os.path.join(save_path_base, dataset, f"{split}.npz")

    data = np.load(data_path)
    x = data["x"]
    y = data["y"]

    extreme_values_binary_tensor = np.zeros_like(y)
    thresholds = np.zeros((y.shape[2], y.shape[3]), dtype=y.dtype)

    # Loop through each node and each direction (inflow, outflow)
    for node_index in range(y.shape[2]):  # y shape is [samples, timesteps, nodes, directions]
        for direction_index in range(y.shape[3]):
            # Get the data for current node and direction
            series = y[:, 0, node_index, direction_index]

            # Calculate the xxth percentile
            threshold = np.percentile(series, xx)
            thresholds[node_index, direction_index] = threshold

            # Find indices where data exceeds the threshold
            extreme_indices = np.where(series > threshold)[0]

            # Update the binary tensor
            extreme_values_binary_tensor[extreme_indices, 0, node_index, direction_index] = 1

    np.savez(out_path, x=x, y=y, evs_95=extreme_values_binary_tensor, threshold_95=thresholds)
